# Phase 3 — Avaliação dupla GNN vs SIR (execução no Colab GPU)

Projeto **BraSNAM 2026** — GNN temporal heterogênea para difusão de popularidade musical.

Este notebook roda o **orquestrador já implementado e testado** `scripts/run_phase3.py`
(T1–T11 da feature `phase-3-evaluation`, 105 testes verdes) numa GPU do Colab, em vez de
localmente em CPU. Motivo: a máquina local tem só 7.6 GB de RAM / CPU-only; o Modo 2
(rollout recursivo, ~47 mil origens no dataset completo) é lento nela, e o prazo de
submissão é 2026-07-11.

**Pré-requisitos que este notebook busca automaticamente:**
1. Código + os 3 artefatos de dados versionados (`hetero_full.pt`, `node_id_map.json`,
   `timeseries.parquet`) — via `git clone`.
2. O checkpoint `grid_best_model.pt` (config W12_h128_l3 da Phase 2) — via Google Drive,
   pasta `music-influence-gnn/phase2_experimentos_v2/` (mesma usada pelo notebook de treino).
3. `results/phase0/sir_params.parquet` — opcional; se ausente, o próprio script refaz o
   fit SIR (mais lento, mas funciona).


## 0. Ambiente — clonar o repositório

Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU (T4)* antes de rodar.
Repo privado: defina `GITHUB_TOKEN` abaixo ou deixe em branco e cole um PAT quando pedido.

In [1]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

# >>> ajuste conforme seu repositório <<<
REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN (vazio = público)

DATA_FILES = [
    "data/processed/graph/hetero_full.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if not IS_COLAB:
    raise RuntimeError("Este notebook é para rodar no Google Colab (runtime GPU). Localmente, use scripts/run_phase3.py diretamente.")

if not Path(REPO_DIR, "pyproject.toml").exists():
    tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
    url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
    print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
    if _clone(url).returncode != 0:
        from getpass import getpass
        tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
        _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
os.chdir(REPO_DIR)
# -e . instala o pacote music_diffusion_gnn (src/ layout) em modo editável — necessário
# porque as células abaixo chamam scripts/run_phase3.py via `!python ...`, um PROCESSO
# SEPARADO do kernel deste notebook, que não herda sys.path.insert() feito em Python puro.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "torch-geometric", "pyarrow"], check=True)

missing = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
print("cwd =", os.getcwd())
if missing:
    raise RuntimeError(f"Faltam dados versionados no repo clonado: {missing}")
print("dados versionados presentes.")

# Sanidade: confirma que o pacote é importável tanto aqui (kernel) quanto em subprocessos
# `!python ...` (as células de execução do script chamam um processo novo).
subprocess.run([sys.executable, "-c", "import music_diffusion_gnn; print('music_diffusion_gnn OK, subprocess')"], check=True)


cwd = /content/music-influence-gnn
dados versionados presentes.


CompletedProcess(args=['/usr/bin/python3', '-c', "import music_diffusion_gnn; print('music_diffusion_gnn OK, subprocess')"], returncode=0)

## 1. Checkpoint da Phase 2 (`grid_best_model.pt`) + SIR cache — via Google Drive

O checkpoint **não** é versionado no git (`.gitignore` exclui `results/*`). Ele foi salvo no
Google Drive pelo notebook de treino (`phase2_pipeline_treino.ipynb`, célula "Persistência no
Drive") em `MyDrive/music-influence-gnn/phase2_experimentos_v2/grid_best_model.pt`.
Este notebook monta o Drive e copia esse arquivo (e o cache do SIR, se existir) para dentro
do repositório clonado, nos mesmos caminhos que `run_phase3.py` espera.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/music-influence-gnn")
CKPT_SRC = DRIVE_ROOT / "phase2_experimentos_v2" / "grid_best_model.pt"
SIR_SRC = DRIVE_ROOT / "phase0" / "sir_params.parquet"

CKPT_DST = Path(REPO_DIR, "results", "phase2_experimentos_v2", "grid_best_model.pt")
SIR_DST = Path(REPO_DIR, "results", "phase0", "sir_params.parquet")
CKPT_DST.parent.mkdir(parents=True, exist_ok=True)
SIR_DST.parent.mkdir(parents=True, exist_ok=True)

if CKPT_SRC.exists():
    import shutil
    shutil.copy(CKPT_SRC, CKPT_DST)
    print(f"checkpoint copiado de {CKPT_SRC}")
else:
    print(f"NÃO encontrado em {CKPT_SRC} — faça upload manual abaixo (célula seguinte).")

if SIR_SRC.exists():
    import shutil
    shutil.copy(SIR_SRC, SIR_DST)
    print(f"cache SIR copiado de {SIR_SRC}")
else:
    print("cache SIR não encontrado no Drive — run_phase3.py refará o fit automaticamente (R0/B).")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoint copiado de /content/drive/MyDrive/music-influence-gnn/phase2_experimentos_v2/grid_best_model.pt
cache SIR não encontrado no Drive — run_phase3.py refará o fit automaticamente (R0/B).


### 1.1 (Fallback) upload manual do checkpoint

Só rode esta célula se a cópia do Drive acima **não** encontrou o arquivo — por exemplo, se o
treino da Phase 2 foi salvo em outro caminho. Selecione `grid_best_model.pt` no seletor de
arquivos.

In [3]:
if not CKPT_DST.exists():
    from google.colab import files
    print("Selecione grid_best_model.pt:")
    uploaded = files.upload()
    fname = next(iter(uploaded))
    CKPT_DST.write_bytes(uploaded[fname])
    print(f"salvo em {CKPT_DST}")
else:
    print("checkpoint já presente, upload manual não necessário:", CKPT_DST)


checkpoint já presente, upload manual não necessário: /content/music-influence-gnn/results/phase2_experimentos_v2/grid_best_model.pt


## 2. Verificar GPU

In [4]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError(
        "GPU NÃO ativada no Colab! Vá em: Ambiente de execução → Alterar tipo de "
        "runtime → GPU (T4), salve e rode tudo de novo."
    )
_p = torch.cuda.get_device_properties(0)
print(f"GPU: {_p.name} | {_p.total_memory / 1e9:.1f} GB")


GPU: Tesla T4 | 15.6 GB


## 3. Smoke test (gate antes da execução completa)

Mesmo gate que `tasks.md` (T11) exige antes de gastar tempo na execução real: roda em poucas
músicas/origens, valida que o pipeline não quebra, cria `results/phase3/*`.

In [5]:
!cd {REPO_DIR} && python -u scripts/run_phase3.py --smoke --device cuda



  R0/A — Load timeseries + graph
  Loaded: (4443760, 5) rows, 6526 music nodes  [15.4s]

  → Smoke mode: 5 songs, 12594 daily rows

  → Smoke weekly: 1706 rows

  R0/B — SIR baseline fits

  → Fitting SIR (parallelized)…
  SIR fit: 10 songs, converged 100.0%  [6.4s]

  R0/C — Load GNN checkpoint + pop_bank
/content/music-influence-gnn/src/music_diffusion_gnn/evaluation/model_io.py:62: UserWarning: Regenerated pop_bank differs from the checkpoint's frozen pop_bank (same shape, different values).
  warnings.warn(

  → Loaded GNN: W=12, hidden=128, layers=3, val_mse=0.000749

  M1/A — GNN free rollout (Mode 1)

  → GNN free rollout cached: 551184 rows

  M1/B — SIR Mode-1 weekly reconstruction

  → SIR Mode-1: 1706 rows  [0.1s]

  M1/C — Compute per-song RMSE → mode1_per_song.parquet

  → mode1_per_song: 3920 rows (songs × models)

  M2/A — Build origins

  → Origins: 10 rows (songs × weeks), stride=4

  M2/B — GNN recursive rollout (Mode 2)

  → GNN recursive cached: 141504 rows

  M2/C

## 4. Execução completa (T12) — dataset e checagem C1–C12

Roda o dataset inteiro (todas as músicas, ~47 mil origens de Modo 2 com stride=4). Na GPU
T4 as convoluções do encoder (`encode_weeks`) rodam em paralelo — deve ser bem mais rápido
que os ~4,6 min/etapa observados localmente em CPU só para o Modo 1.

In [6]:
# Execução completa. LIMPA results/phase3 ANTES de rodar: o smoke (célula 3) deixou
# gnn_free_rollout / gnn_recursive_rollout / interpretability em cache, e run_phase3.py
# os REUSA ("... cached") — sem limpar, o full herda a GNN do smoke (3-5 músicas) e só o
# SIR sai completo. O cache do SIR full fica em results/phase0 (NÃO é apagado por isto),
# então só a GNN é recomputada do zero, sobre o dataset inteiro.
!cd {REPO_DIR} && rm -rf results/phase3 && python -u scripts/run_phase3.py --device cuda



  R0/A — Load timeseries + graph
  Loaded: (4443760, 5) rows, 6526 music nodes  [14.8s]

  R0/B — SIR baseline fits

  → SIR cached: 3962 fits  [results/phase0/sir_params.parquet]

  R0/C — Load GNN checkpoint + pop_bank

  → Loaded GNN: W=12, hidden=128, layers=3, val_mse=0.000749

  M1/A — GNN free rollout (Mode 1)
gnn_rollout_free: 3910 (song,chart) rolled out, 2 excluded (span<2), 94 seed-clamped (span<=W).

  → GNN free rollout: 551184 rows  [2.5min]

  M1/B — SIR Mode-1 weekly reconstruction

  → SIR Mode-1: 597672 rows  [53.7s]

  M1/C — Compute per-song RMSE → mode1_per_song.parquet

  → mode1_per_song: 7820 rows (songs × models)

  M2/A — Build origins

  → Origins: 47168 rows (songs × weeks), stride=4

  M2/B — GNN recursive rollout (Mode 2)

  → GNN recursive: 141504 rows  [8.6min]

  M2/C — SIR causal refit + persistence baseline

  → SIR Mode-2: 141504 rows, 1758 non-convergent  [62.3min]

  M2/D — Assemble mode2_horizons.parquet

  → mode2_horizons: 424512 rows

  M2/E —

## 5. Persistir resultados no Drive

Copia `results/phase3/` inteiro para o Drive (sobrevive ao encerramento da sessão do Colab).

In [7]:
import shutil, pandas as pd

src_dir = Path(REPO_DIR, "results", "phase3")

# TRAVA anti-smoke: só persiste se a GNN cobrir o dataset inteiro. Impede repetir o bug em
# que um cache do smoke (3 músicas) foi copiado pro Drive como se fosse o full. Se disparar,
# rode a célula 12 (que agora limpa results/phase3) antes de persistir de novo.
_m2 = pd.read_parquet(src_dir / "mode2_horizons.parquet")
_gnn_songs = int(_m2.loc[_m2.model == "gnn", "song_id"].nunique())
assert _gnn_songs > 100, (
    f"ABORTADO: a GNN cobre só {_gnn_songs} músicas no mode2_horizons — isto é um run "
    "SMOKE/cache, não o full. Rode a célula 12 (limpa results/phase3) e tente de novo."
)

DRIVE_PHASE3 = DRIVE_ROOT / "phase3_resultados"
DRIVE_PHASE3.mkdir(parents=True, exist_ok=True)
for f in src_dir.iterdir():
    shutil.copy(f, DRIVE_PHASE3 / f.name)
print(f"[OK full: GNN em {_gnn_songs} músicas] resultados copiados para {DRIVE_PHASE3}")

print("\n" + "=" * 60)
print((src_dir / "summary.md").read_text())


[OK full: GNN em 1950 músicas] resultados copiados para /content/drive/MyDrive/music-influence-gnn/phase3_resultados

# Phase 3 — GNN vs SIR Dual Evaluation



## Mode 1 — Free Rollout (Reconstruction)

| Regime | GNN RMSE | SIR RMSE | GNN RMSE (on-chart) | SIR RMSE (on-chart) |
|--------|----------|----------|---------------------|---------------------|
| Virality | 0.02660 | 0.01411 | 0.19151 | 0.17604 |
| Success | 0.06321 | 0.03422 | 0.16373 | 0.12588 |

## Mode 2 — Recursive Rollout (k-step Forecasting)

SIR non-convergent/short-history exclusions: 1758

| Model | k | Regime | RMSE | Dir. Acc. | n |
|-------|---|--------|------|-----------|---|
| gnn | 1 | viral50 | 0.02612 | 0.018 | 23584 |
| gnn | 1 | top200 | 0.02381 | 0.062 | 23584 |
| gnn | 2 | viral50 | 0.03455 | 0.020 | 23584 |
| gnn | 2 | top200 | 0.03699 | 0.066 | 23584 |
| gnn | 4 | viral50 | 0.04440 | 0.025 | 23584 |
| gnn | 4 | top200 | 0.05133 | 0.073 | 23584 |
| sir | 1 | viral50 | 0.03695 | 0.012 | 23291 |
| sir | 1

## 6. Próximos passos (a fazer localmente, após baixar os resultados do Drive)

1. Baixar `results/phase3/*` do Drive (`music-influence-gnn/phase3_resultados/`) para o
   repositório local, na mesma pasta `results/phase3/`.
2. Revisar `summary.md` e o checklist C1–C12 (a etapa "Review summary.md" do plano de
   continuação da Phase 3).
3. Registrar os números finais desta execução em `.specs/project/STATE.md` (os 3 fixes de
   device/memória já estão commitados no `main` — commit `23a5fb5`).